# Constrained optimisation

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps

In [ ]:
import numpy as np
from scipy.optimize import minimize
from compas.geometry import Box
from compas_notebook import Viewer
from compas_notebook.config import Config

## Minimise area at fixed volume

In [ ]:
volume = 3.0
height_limit = 1.0


def area(d):
    x, y, z = d
    return 2 * (x * y + x * z + y * z)


def area_gradient(d):
    x, y, z = d
    return np.array([2 * (y + z), 2 * (x + z), 2 * (x + y)])


def volume_constraint(d):
    return np.prod(d) - volume


def volume_gradient(d):
    x, y, z = d
    return np.array([y * z, x * z, x * y])

## SLSQP

In [ ]:
result = minimize(
    area,
    [volume, 1.0, 1.0],
    jac=area_gradient,
    method="SLSQP",
    bounds=[(0.1, None), (0.1, None), (0.1, height_limit)],
    constraints=[{"type": "eq", "fun": volume_constraint, "jac": volume_gradient}],
)
result.x, result.fun

## Scene

In [ ]:
config = Config()
config.view.width, config.view.height = 700, 360
config.view.camera.position = [4, -6, 3]
viewer = Viewer(config=config)
viewer.scene.add(Box(*result.x).to_mesh(), color="#5552bb", show_edges=True)
viewer.show()